# Source-domain and genre-specific sentiment training

Train the shared source-domain RoBERTa model and independently fine-tune its genre-specific experts. Only review text and sentiment labels are used. Provide the data described in the repository README and select a GPU before training. Configure a new output directory to preserve existing checkpoints.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

repo = Path("/content/genre-expert-ensembles") if Path("/content").exists() else Path.cwd()
if not (repo / "common.py").is_file():
    if not Path("/content").exists():
        raise RuntimeError("Run this notebook from the cloned repository root.")
    subprocess.run([
        "git", "clone",
        "https://github.com/Jun-Wang0/Domain-level-learning-and-genre-specific-expert-ensembles.git",
        str(repo),
    ], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

os.environ["ENSEMBLE_ROOT"] = "/content/drive/MyDrive/PhD"
os.environ["ENSEMBLE_OUTPUT"] = str(Path(os.environ["ENSEMBLE_ROOT"]) / "genre_ensemble_runs")
print("Repository:", repo)
print("New experiment outputs:", os.environ["ENSEMBLE_OUTPUT"])


## Run configuration

The first allocation trains All. The second trains the ten independently initialized genre experts for each source. Optional ablation allocations are `0+4`, `1+3`, and `3+1`. Existing destinations are not overwritten.

In [ ]:
SOURCES = ["Book", "Game", "Movie"]
ALLOCATIONS = ["4+0", "2+2"]
BATCH_SIZE = 32
SEED = 42

In [ ]:
from train_sentiment import train_source

for source in SOURCES:
    for allocation in ALLOCATIONS:
        train_source(source, allocation, batch_size=BATCH_SIZE, seed=SEED)